# TARDIS - Construction du Modèle de Prédiction

Ce notebook contient le processus de préparation des données, l'entraînement d'un modèle de régression pour prédire les retards de trains de la SNCF, et son évaluation.

In [8]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Chargement du dataset nettoyé
df = pd.read_csv("cleaned_dataset.csv")
df['Date'] = pd.to_datetime(df['Date'])

## 1. Préparation des Caractéristiques (Feature Engineering) 

Nous extrayons le jour de la semaine et sélectionnons les variables d'entrée pertinentes comme les gares et le service.

In [9]:
df['day_of_week'] = df['Date'].dt.dayofweek

# Définition de la cible (retard à l'arrivée en minutes)
target = "Retard moyen de tous les trains à l'arrivée"

# Sélection des variables d'entrée 
features_cat = ["Gare de départ", "Gare d'arrivée", "Service"]
features_num = ["Année", "Mois", "day_of_week"]

X = df[features_cat + features_num]
y = df[target]

# Division Train/Test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Pipeline et Entraînement

Nous créons un pipeline qui gère l'encodage des variables catégorielles et l'entraînement d'un modèle Random Forest.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
    ]
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=100, random_state=42)),
    ]
)

model_pipeline.fit(X_train, y_train)

## 3. Évaluation du Modèle

Nous mesurons la performance via la MAE, le RMSE et le R². Nous vérifions également que le modèle est plus performant que la baseline (prédiction de la moyenne).

In [ ]:
y_pred = model_pipeline.predict(X_test)

# Calcul des métriques
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Baseline (moyenne)
baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))

print(f"MAE du modèle : {mae:.2f} min")
print(f"MAE de la Baseline : {baseline_mae:.2f} min")
print(f"RMSE : {rmse:.2f} min")
print(f"R² Score : {r2:.2f}")

MAE du modèle : 2.02 min
MAE de la Baseline : 3.08 min
RMSE : 3.47 min
R² Score : 0.39


## 4. Exportation du Modèle 

Sauvegarde pour intégration dans le dashboard Streamlit.

In [ ]:
joblib.dump(model_pipeline, "model.joblib")

['model.joblib']